In [3]:
import pandas as pd
import scanpy as sc
import numpy as np
import anndata as ad
import os
import colorcet as cc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import harmonypy as hm
import scrublet as scr
import traceback
from collections import Counter
from sklearn.decomposition import PCA
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import re
from matplotlib.patches import Patch
from PIL import Image
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
%matplotlib inline

# Define Working Directory
working_dic = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_3month/"
os.chdir(working_dic)
sc.settings.verbosity = 1
sc.logging.print_header()

# Professional Plotting Parameters
# sc.set_figure_params( dpi=100,  dpi_save=300,  vector_friendly=True, format='pdf',  transparent=True, facecolor='white' )

dimplot_color = mcolors.LinearSegmentedColormap.from_list("custom_cmap", ["#E5E5E5", "#4C4DB7"])

COND_ORDER = [
    "Control",
    "Prematurely-terminated",
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
]

COND_PALETTE = {
    "Control": "#EFE48E",
    "3month-protocol": "skyblue",
    "6month-protocol": "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV": "#F0C096",
    "Rejection": "#E09E9E",
}


# Saving pseudobulk counts for edgeR

In [5]:
# input paths
ADATA_PATH = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/global_object_assembly/objects/global_X_UMAP_Added_PCMV_score_updated_celltype_cellstate_annotated_preclinical_object_RAW_v3.h5ad"
 
# Output paths
BASE_OUT_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_3month"
 
#  Metadata columns
SAMPLE_COL     = "Sample_ID"
CONDITION_COL  = "Condition"
CELL_TYPE_COL  = "cell_type"
CELL_STATE_COL = "cell_state"
 
# D. Cell types to skip (no cell_state annotations)
SKIP_CELL_TYPES = [
    "MC",
    "NC",
    "EC_papio",
]
 
# Loading adata

PB_DIR = os.path.join(BASE_OUT_DIR, "pseudobulks")
os.makedirs(PB_DIR, exist_ok=True)
 
print("Loading adata...")
adata = sc.read(ADATA_PATH)
print(f"  Shape: {adata.shape[0]} cells x {adata.shape[1]} genes")
 
# export metadata
 
print("Exporting meta_info.csv...")
meta_df = adata.obs[[SAMPLE_COL, CONDITION_COL]].drop_duplicates().reset_index(drop=True)
meta_df.to_csv(os.path.join(BASE_OUT_DIR, "meta_info.csv"), index=False)
print(f"  {len(meta_df)} unique samples saved.")
 
# generating pseudobulks per cell_state for edger later on
 
cell_types = sorted(adata.obs[CELL_TYPE_COL].unique().tolist())
print(f"\nFound {len(cell_types)} cell types: {cell_types}")
print(f"Skipping: {SKIP_CELL_TYPES}\n")
 
total_files = 0
 
for ct in cell_types:
 
    # Skip cell types without cell_state annotations
    if ct in SKIP_CELL_TYPES:
        print(f"[SKIP] {ct} — no cell_state annotations")
        continue
 
    adata_ct = adata[adata.obs[CELL_TYPE_COL] == ct]
    cell_states = sorted(adata_ct.obs[CELL_STATE_COL].dropna().unique().tolist())
 
    if len(cell_states) == 0:
        print(f"[SKIP] {ct} — no cell_states found in data")
        continue
 
    print(f"[{ct}] Processing {len(cell_states)} cell states...")
 
    for cs in cell_states:
        adata_cs = adata_ct[adata_ct.obs[CELL_STATE_COL] == cs]
 
        if adata_cs.n_obs == 0:
            continue
 
        # Build pseudobulk: sum raw counts per sample
        pb_df = pd.DataFrame(index=adata_cs.var_names)
 
        samples = sorted(adata_cs.obs[SAMPLE_COL].unique().tolist())
 
        for sample in samples:
            sample_adata = adata_cs[adata_cs.obs[SAMPLE_COL] == sample]
 
            if sp.issparse(sample_adata.X):
                sample_sum = np.array(sample_adata.X.sum(axis=0)).flatten()
            else:
                sample_sum = np.array(sample_adata.X.sum(axis=0)).flatten()
 
            pb_df[sample] = sample_sum
 
        # Save with ___ delimiter so R can parse cell_type and cell_state
        safe_ct = str(ct).replace("/", "_")
        safe_cs = str(cs).replace("/", "_")
        out_path = os.path.join(PB_DIR, f"{safe_ct}___{safe_cs}.csv")
        pb_df.to_csv(out_path)
        total_files += 1
 
    print(f"  -> {len(cell_states)} files saved for {ct}")
 
print(f"\nDone! {total_files} pseudobulk CSVs saved to: {PB_DIR}")
 

Loading adata...
  Shape: 233200 cells x 53826 genes
Exporting meta_info.csv...
  26 unique samples saved.

Found 10 cell types: ['CM', 'EC', 'EC_papio', 'FB', 'Lymph_papio', 'Lymph_sus', 'MC', 'Mye_papio', 'Mye_sus', 'NC']
Skipping: ['MC', 'NC', 'EC_papio']

[CM] Processing 7 cell states...
  -> 7 files saved for CM
[EC] Processing 9 cell states...
  -> 9 files saved for EC
[SKIP] EC_papio — no cell_state annotations
[FB] Processing 8 cell states...
  -> 8 files saved for FB
[Lymph_papio] Processing 10 cell states...
  -> 10 files saved for Lymph_papio
[Lymph_sus] Processing 2 cell states...
  -> 2 files saved for Lymph_sus
[SKIP] MC — no cell_state annotations
[Mye_papio] Processing 9 cell states...
  -> 9 files saved for Mye_papio
[Mye_sus] Processing 10 cell states...
  -> 10 files saved for Mye_sus
[SKIP] NC — no cell_state annotations

Done! 55 pseudobulk CSVs saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_3month/pseudobul